# Chapitre 6 · Le neurone et le réseau

Notebook du chapitre 6 de *Construire un LLM de zéro* · Partie II « Construire le cerveau ».

**Comment travailler.** D'abord la leçon : tout le code du chapitre, complet et
exécutable de bout en bout, du neurone unique du chapitre 4 au MLP qui plie sa
frontière de décision. Lis, exécute, modifie pour voir. À la fin, la section
**Exercices** : trois défis à trous, du plus simple au plus costaud, validés
par des `assert`.

## 1. La dernière boîte noire de la notice

Bienvenue dans la Partie II. Rouvre une dernière fois la notice de MiniLM (chapitre 1) :

```python
self.reseau = nn.Sequential(
    nn.Linear(block_size * 24, 192),
    nn.Tanh(),
    nn.Linear(192, vocab_size),
)
```

Deux questions ouvertes depuis le premier jour : pourquoi **deux** étages `Linear`
au lieu d'un seul ? Et que vient faire ce `nn.Tanh()` glissé entre les deux ?
Ce chapitre répond aux deux, sur un problème à deux nombres d'entrée, assez petit
pour dessiner chaque frontière.

## 2. Le neurone, officiellement

Tu as déjà construit un neurone, au chapitre 4, pour le baptême de ton moteur.
Il **dose** chaque entrée par un poids, **décale** le résultat d'un biais, et
fait éventuellement passer le tout par une fonction (le `tanh`, dont le rôle
exact est l'affaire de la section 6). Côté espace : un neurone seul trace une
**droite** et prend parti de part et d'autre.

In [ ]:
import torch

# La recette du chapitre 4, en tenseurs PyTorch : doser, decaler, plier.
x1, x2 = torch.tensor(2.0), torch.tensor(0.5)     # les entrees
w1, w2 = torch.tensor(-3.0), torch.tensor(1.0)    # les poids (la recette)
b = torch.tensor(6.5)                             # le biais

o = (w1*x1 + w2*x2 + b).tanh()                    # le neurone, en une ligne
print(f"score = w.x + b = {(w1*x1 + w2*x2 + b).item():.2f}")
print(f"o = tanh(score) = {o.item():.4f}")

## 3. Les courses de Sètondji

### 3.1 Le carnet

Sètondji, le conducteur de zémidjan du chapitre 3, connaît désormais son tarif.
Son nouveau problème : **quelles courses accepter ?** Une course est rentable si
la destination est un coin où il retrouve tout de suite un client pour le retour
(autour du grand marché où il stationne, ou autour de l'aéroport) ; partout
ailleurs, il rentre à vide.

Chaque course de son carnet : une destination `(km est-ouest, km nord-sud)`
mesurée depuis son stand, et un verdict `1` (rentable) ou `0` (à perte).

**Honnêteté oblige** : on fabrique ce carnet nous-mêmes, avec une règle cachée
(deux zones circulaires). Avantage de ce choix, le même que dans les vrais labos
quand ils testent un modèle sur des données synthétiques : on connaît la vérité,
donc on peut vérifier exactement ce que le réseau a appris.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# 400 courses : destination (km est-ouest, km nord-sud) depuis le stand.
n_courses = 400
destinations = torch.rand(n_courses, 2)
destinations[:, 0] = destinations[:, 0] * 12.0 - 8.0   # est-ouest : -8 .. +4 km
destinations[:, 1] = destinations[:, 1] * 8.0 - 4.0    # nord-sud  : -4 .. +4 km

# La regle cachee : rentable si la destination est a moins de 2.6 km du stand
# (le grand marche) OU a moins de 2.2 km de l'aeroport, situe en (-5.5, -2.0).
dist_marche = (destinations ** 2).sum(dim=1).sqrt()
dist_aeroport = ((destinations[:, 0] + 5.5) ** 2 + (destinations[:, 1] + 2.0) ** 2).sqrt()
cibles = ((dist_marche < 2.6) | (dist_aeroport < 2.2)).long()   # 1 = rentable, 0 = a perte

print(f"destinations : shape {tuple(destinations.shape)}  (400 courses, 2 caracteristiques)")
print(f"cibles       : shape {tuple(cibles.shape)}   (un verdict par course)")
print(f"courses rentables : {cibles.sum().item()} / {n_courses}")
print()
for i in range(4):
    x, y = destinations[i].tolist()
    print(f"course {i} : ({x:5.2f} km, {y:5.2f} km)  ->  {'rentable' if cibles[i] else 'a perte'}")


La carte du carnet. Chaque point est une course ; la règle cachée (les deux
zones) est dessinée en pointillés. C'est **elle** que le réseau devra retrouver,
sans jamais la voir : il ne verra que les points.


In [ ]:
import matplotlib.pyplot as plt

def carte(titre):
    fig, ax = plt.subplots(figsize=(7, 4.6))
    ok = cibles == 1
    ax.scatter(destinations[ok, 0], destinations[ok, 1], s=14, c="#5F7A52", label="rentable")
    ax.scatter(destinations[~ok, 0], destinations[~ok, 1], s=14, c="#C46A4B", marker="x", label="a perte")
    for (cx, cy, r) in [(0.0, 0.0, 2.6), (-5.5, -2.0, 2.2)]:
        ax.add_patch(plt.Circle((cx, cy), r, fill=False, ls="--", color="#7A726A"))
    ax.set_xlabel("km est-ouest"); ax.set_ylabel("km nord-sud")
    ax.set_xlim(-8, 4); ax.set_ylim(-4, 4); ax.set_aspect("equal")
    ax.legend(loc="upper right"); ax.set_title(titre)
    return ax

carte("Le carnet de Setondji : 400 courses, deux zones rentables")
plt.show()


### 3.2 La couche : des neurones en parallèle

Une **couche**, c'est plusieurs neurones branchés en parallèle sur les mêmes
entrées, et tu sais depuis le chapitre 2 que ces sommes pondérées empilées sont
**un produit matriciel**. Conventions de PyTorch (celles de la notice de MiniLM) :

- `entrees` : shape `(n, 2)`, une ligne par course, une colonne par caractéristique ;
- `W` : shape `(nb_neurones, 2)`, une **ligne par neurone** (sa recette) ;
- `b` : shape `(nb_neurones,)`, un biais par neurone.

La sortie doit avoir la shape `(n, nb_neurones)` : un score par neurone et par
course. Il faut donc transposer `W` pour que les dimensions s'emboîtent :
`(n, 2) @ (2, nb_neurones)`. Une seule ligne de calcul, que tu réécriras de
tête à l'exercice 1.

In [ ]:
def forward_couche(entrees, W, b):
    """La couche : chaque ligne de W est la recette d'un neurone.

    entrees : (n, e)   W : (s, e)   b : (s,)   ->   sortie : (n, s)
    """
    return entrees @ W.T + b


# Demonstration sur des valeurs simples, verifiables de tete.
entrees_demo = torch.tensor([[2.0, 3.0],
                             [1.0, -1.0]])
W_demo = torch.tensor([[1.0, 0.0],     # neurone 1 : recopie x1
                       [0.0, 1.0],     # neurone 2 : recopie x2
                       [1.0, 1.0],     # neurone 3 : x1 + x2
                       [0.5, -0.5]])   # neurone 4 : (x1 - x2) / 2
b_demo = torch.tensor([0.0, 0.0, 1.0, 2.0])

sortie_demo = forward_couche(entrees_demo, W_demo, b_demo)
print(f"shape : {tuple(entrees_demo.shape)} @ {tuple(W_demo.T.shape)} + {tuple(b_demo.shape)} -> {tuple(sortie_demo.shape)}")
print(sortie_demo)


### `nn.Linear` fait exactement ça

La couche de PyTorch, `nn.Linear`, c'est ta fonction emballée dans un objet :
elle crée `W` et `b` toute seule (avec `requires_grad=True` d'office, comme les
feuilles de ton moteur du chapitre 4) et les range dans `.weight` et `.bias`.
On copie tes matrices dedans et on compare : mêmes nombres.


In [ ]:
couche = nn.Linear(2, 4)          # 2 entrees -> 4 neurones
print(f"couche.weight : shape {tuple(couche.weight.shape)}   (une ligne par neurone)")
print(f"couche.bias   : shape {tuple(couche.bias.shape)}     (un biais par neurone)")
print(f"parametres    : {sum(p.numel() for p in couche.parameters())}  (4 x 2 poids + 4 biais)")

with torch.no_grad():             # on modifie les poids sans que l'autograd n'enregistre
    couche.weight.copy_(W_demo)
    couche.bias.copy_(b_demo)

assert torch.allclose(couche(entrees_demo), forward_couche(entrees_demo, W_demo, b_demo)), \
    "nn.Linear et ta fonction devraient donner les memes nombres"
print("nn.Linear(2, 4) == ta forward_couche : memes nombres, au bit pres.")


### Premier candidat : un seul étage linéaire

Deux classes (à perte / rentable), donc **deux scores** en sortie, exactement comme
MiniLM sortait 81 scores : `nn.Linear(2, 2)`. La loss est la cross-entropy du
chapitre 5, qui contient déjà le softmax.

Avant tout entraînement, lisons la loss de départ avec les yeux du chapitre 5 :
le hasard pur sur 2 candidats vaudrait `-log(1/2) = log(2) ≈ 0.693`.


In [ ]:
torch.manual_seed(1)
modele_lineaire = nn.Linear(2, 2)     # 2 caracteristiques -> 2 scores (a perte, rentable)

with torch.no_grad():
    loss_depart = F.cross_entropy(modele_lineaire(destinations), cibles).item()

print(f"loss de depart : {loss_depart:.4f}")
print(f"hasard pur sur 2 candidats : log(2) = {math.log(2):.4f}")
print("1.77 > 0.69 : des poids aleatoires ne donnent pas l'uniforme, ils sont")
print("confiants ET faux. La descente va corriger ca.")


## 4. La boucle d'entraînement, propre

Le refrain du livre, en cinq lignes de PyTorch. Chaque ligne, tu l'as construite :

1. `logits = modele(...)` : le **forward**, l'aller du chapitre 4 ;
2. `loss = F.cross_entropy(...)` : la mesure d'erreur du chapitre 5 ;
3. `optimiseur.zero_grad()` : la remise à zéro des compteurs de gradient (le `+=` du chapitre 4) ;
4. `loss.backward()` : le retour, ton moteur du chapitre 4 ;
5. `optimiseur.step()` : le pas de descente du chapitre 3, sur tous les poids d'un coup.

Grave l'ordre des trois dernières : `zero_grad`, `backward`, `step`.
Tu réécriras ces cinq lignes de tête à l'exercice 2.


In [ ]:
optimiseur = torch.optim.SGD(modele_lineaire.parameters(), lr=0.2)

for etape in range(10001):
    logits = modele_lineaire(destinations)          # 1. forward : (400, 2) -> (400, 2)
    loss = F.cross_entropy(logits, cibles)          # 2. mesurer l'erreur (chapitre 5)
    optimiseur.zero_grad()                          # 3. remettre les compteurs a zero
    loss.backward()                                 # 4. le retour : tous les gradients
    optimiseur.step()                               # 5. le pas de descente (chapitre 3)
    if etape % 2000 == 0:
        print(f"etape {etape:5d} | loss = {loss.item():.4f}")


### Les deux garde-fous de la lecture

Pour **interroger** un modèle au lieu de l'entraîner, deux réflexes promis au
chapitre 1 :

- `modele.eval()` : passe le modèle en mode lecture. Sans effet sur nos modèles
  d'aujourd'hui, mais certaines pièces qu'on croisera plus tard (le dropout au
  chapitre 11) se comportent différemment à l'entraînement et à la lecture ;
- `torch.no_grad()` : suspend l'enregistrement du graphe de calcul (les `_prev`
  de ton moteur du chapitre 4). Aucun `backward()` ne suivra, inutile de payer
  la mémoire de la trace.

Mesurons le modèle avec ces garde-fous : son **accuracy**, le taux de bonnes réponses.

In [ ]:
def accuracy(modele):
    """Le taux de bonnes reponses sur les 400 courses du carnet."""
    modele.eval()                                   # garde-fou 1 : mode lecture
    with torch.no_grad():                           # garde-fou 2 : pas de graphe
        predictions = modele(destinations).argmax(dim=1)
    modele.train()
    return (predictions == cibles).float().mean().item()

acc_lineaire = accuracy(modele_lineaire)
loss_finale = F.cross_entropy(modele_lineaire(destinations), cibles).item()
print(f"loss finale : {loss_finale:.4f} | accuracy : {acc_lineaire:.3f}")
print("63,2 % la ou repondre 'a perte' les yeux fermes donne deja 62,5 % :")
print("un plafond, pas une victoire.")

## 5. Le mur : une droite ne suffit pas

### 5.1 La frontière, dessinée

63,2 % de bonnes réponses, et la loss est clouée à 0.6235 depuis l'étape 2000 :
le modèle a fini d'apprendre **tout ce qu'il peut apprendre**. La carte montre
pourquoi : sa frontière de décision est une droite, et aucune droite ne découpe
deux poches rondes.


In [ ]:
import numpy as np

def carte_frontiere(modele, titre):
    """Trace la carte du carnet + la frontiere de decision du modele."""
    gx, gy = np.meshgrid(np.linspace(-8, 4, 241), np.linspace(-4, 4, 161))
    grille = torch.tensor(np.stack([gx.ravel(), gy.ravel()], axis=1), dtype=torch.float32)
    modele.eval()
    with torch.no_grad():
        p = F.softmax(modele(grille), dim=1)[:, 1].reshape(gx.shape).numpy()
    modele.train()
    ax = carte(titre)
    ax.contour(gx, gy, p, levels=[0.5], colors="#3A45C9", linewidths=2)
    plt.show()

carte_frontiere(modele_lineaire, f"Un etage lineaire : une droite ({accuracy(modele_lineaire):.1%})")


### 5.2 Le cas qui échoue : empiler deux couches SANS activation

Le réflexe naïf : « une couche ne suffit pas ? Mettons-en deux. »
`Linear(2, 16)` puis `Linear(16, 2)` : 16 neurones intermédiaires, 82 paramètres
au lieu de 6. Regarde les chiffres, puis la preuve algébrique : ce gros modèle
est **exactement** une droite déguisée.


In [ ]:
def entrainer(modele, lr=0.2, n_etapes=10001, journal=0):
    """La meme boucle de cinq lignes que la section 4, emballee en fonction."""
    optimiseur = torch.optim.SGD(modele.parameters(), lr=lr)
    for etape in range(n_etapes):
        logits = modele(destinations)
        loss = F.cross_entropy(logits, cibles)
        optimiseur.zero_grad()
        loss.backward()
        optimiseur.step()
        if journal and etape % journal == 0:
            print(f"etape {etape:5d} | loss = {loss.item():.4f}")
    return loss.item()

torch.manual_seed(1)
empile = nn.Sequential(nn.Linear(2, 16), nn.Linear(16, 2))   # DEUX couches, RIEN entre elles
loss_empile = entrainer(empile)

print(f"parametres : {sum(p.numel() for p in empile.parameters())} (contre 6 pour l'etage unique)")
print(f"loss finale : {loss_empile:.4f} | accuracy : {accuracy(empile):.3f}")
print("Exactement les memes 0.6235 et 63.2 % que l'etage unique. Zero progres.")


In [ ]:
# La preuve algebrique : les deux couches fusionnent en une seule.
# W2 (W1 x + b1) + b2  =  (W2 W1) x + (W2 b1 + b2)  =  W_fusion x + b_fusion
W1, b1 = empile[0].weight, empile[0].bias
W2, b2 = empile[1].weight, empile[1].bias

fusion = nn.Linear(2, 2)
with torch.no_grad():
    fusion.weight.copy_(W2 @ W1)          # (2, 16) @ (16, 2) -> (2, 2)
    fusion.bias.copy_(W2 @ b1 + b2)

ecart = (empile(destinations) - fusion(destinations)).abs().max().item()
print(f"ecart maximal entre les deux modeles, sur les 400 courses : {ecart:.2e}")
print("Les 82 parametres de l'empilement = une petite couche (2, 2) deguisee.")


## 6. L'activation : la pièce qui plie

### 6.1 ReLU et le forward d'un MLP, à la main

La pièce manquante : une fonction qui **plie**, glissée entre les deux couches,
appliquée nombre par nombre. **ReLU** : `max(0, x)`, zéro à gauche, la pente
inchangée à droite. Un coude. Écrivons le pli, puis le forward complet d'un
MLP : couche, pli, couche (tu les réécriras de tête à l'exercice 3).

In [ ]:
def relu_maison(t):
    """max(0, x), applique element par element : le pli."""
    return torch.clamp(t, min=0.0)


def forward_mlp(entrees, W1, b1, W2, b2):
    """Couche -> pli -> couche : le plus petit MLP du monde."""
    cachee = relu_maison(forward_couche(entrees, W1, b1))
    return forward_couche(cachee, W2, b2)


# Demonstration, verifiable a la main.
e = torch.tensor([[1.0, 2.0], [-2.0, 0.5]])
W1_demo = torch.tensor([[1.0, 0.0], [-1.0, 1.0]])
b1_demo = torch.tensor([0.0, 1.0])
W2_demo = torch.tensor([[2.0, -1.0]])
b2_demo = torch.tensor([0.5])
sortie_mlp = forward_mlp(e, W1_demo, b1_demo, W2_demo, b2_demo)
print(sortie_mlp)


In [ ]:
# Verification pas a pas, a la main.
assert torch.equal(relu_maison(torch.tensor([-3.0, 0.0, 2.5])), torch.tensor([0.0, 0.0, 2.5])), \
    "relu_maison doit ecraser les negatifs a zero et laisser passer le reste"

# A la main : pre-activation [[1, 2], [-2, 3.5]] -> relu [[1, 2], [0, 3.5]]
# -> sortie [2*1 - 1*2 + 0.5, 2*0 - 1*3.5 + 0.5] = [0.5, -3.0]
attendu_mlp = torch.tensor([[0.5], [-3.0]])
assert sortie_mlp is not None and torch.allclose(sortie_mlp, attendu_mlp), \
    "le forward du MLP ne colle pas : couche -> pli -> couche, dans cet ordre"
print("Le compte est bon : couche -> pli -> couche, verifie a la main.")


### 6.2 Le MLP : couche, pli, couche

Le même modèle en PyTorch : `nn.Sequential` enchaîne couche, pli, couche
(c'est la structure exacte du `reseau` de MiniLM au chapitre 1, tanh à la place
de ReLU). Même boucle, mêmes données, 82 paramètres comme l'empilement raté de
la section 5.2. Une seule différence : le pli.


In [ ]:
torch.manual_seed(1)
mlp = nn.Sequential(
    nn.Linear(2, 16),    # 2 caracteristiques -> 16 neurones caches
    nn.ReLU(),           # le pli (aucun parametre)
    nn.Linear(16, 2),    # 16 -> 2 scores
)
print(f"parametres : {sum(p.numel() for p in mlp.parameters())}")

with torch.no_grad():
    print(f"loss de depart : {F.cross_entropy(mlp(destinations), cibles).item():.4f}"
          f"  (proche de log(2) = {math.log(2):.4f} : le hasard sur 2 candidats)")

entrainer(mlp, n_etapes=20001, journal=4000)
print(f"accuracy : {accuracy(mlp):.3f}  ({int(accuracy(mlp) * n_courses)} courses sur {n_courses})")


### 6.3 La frontière se plie

D'où vient la victoire ? Chaque neurone caché trace sa droite et son coude ;
la couche de sortie les combine. Plus de neurones, plus de coudes, une frontière
plus fine. Regarde-la se plier quand on passe de 2 à 4 puis 16 neurones cachés.


In [ ]:
mlps = {}
for h in [2, 4, 16]:
    torch.manual_seed(1)
    m = nn.Sequential(nn.Linear(2, h), nn.ReLU(), nn.Linear(h, 2))
    entrainer(m, n_etapes=20001)
    mlps[h] = m
    n_par = sum(p.numel() for p in m.parameters())
    print(f"h = {h:2d} neurones caches | {n_par:3d} parametres | accuracy = {accuracy(m):.3f}")

for h, m in mlps.items():
    carte_frontiere(m, f"MLP a {h} neurones caches ({accuracy(m):.1%})")


### Interroger le modèle

Interrogeons le MLP comme Sètondji le ferait, destination par destination, avec
les deux garde-fous de la section 4 et le softmax du chapitre 5 : une probabilité
de rentabilité. Savoure la troisième ligne, en bordure exacte de zone : le modèle
a appris où passe sa propre incertitude.

In [ ]:
courses_test = torch.tensor([
    [1.5, 0.5],      # a deux pas du marche
    [-5.0, -1.5],    # zone aeroport
    [2.4, 0.9],      # en bordure exacte de la zone du marche
    [3.0, 3.5],      # loin de tout
])

mlp.eval()
with torch.no_grad():
    probas = F.softmax(mlp(courses_test), dim=1)

for (x, y), p in zip(courses_test.tolist(), probas):
    print(f"destination ({x:5.2f}, {y:5.2f})  ->  p(rentable) = {p[1].item():.3f}")


### Bonus · le second cas qui échoue : l'oubli de `zero_grad`

Le chapitre 4 t'a montré pourquoi les gradients **s'accumulent** (`+=`). Voici ce
que ça coûte en vrai : deux entraînements identiques de 301 étapes, l'un sans la
remise à zéro, l'autre avec. Sans `zero_grad`, chaque pas s'appuie sur la somme
de TOUS les gradients passés : la loss fait du yo-yo et le modèle finit derrière.
Et le plus sournois : aucun message d'erreur, nulle part.


In [ ]:
torch.manual_seed(1)
oublieux = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
optimiseur = torch.optim.SGD(oublieux.parameters(), lr=0.2)
print("SANS zero_grad :")
for etape in range(301):
    loss = F.cross_entropy(oublieux(destinations), cibles)
    loss.backward()                  # les gradients s'ADDITIONNENT a ceux d'avant
    optimiseur.step()
    if etape % 60 == 0:
        print(f"  etape {etape:3d} | loss = {loss.item():.4f}")
print(f"  accuracy : {accuracy(oublieux):.3f}")

torch.manual_seed(1)
propre = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
loss_finale = entrainer(propre, n_etapes=301, journal=60)
print(f"AVEC zero_grad : accuracy : {accuracy(propre):.3f}")


## 7. Relire la notice de MiniLM

Retour aux 40 lignes du chapitre 1. Le `reseau` de MiniLM :

```python
nn.Sequential(
    nn.Linear(block_size * 24, 192),   # 384 entrees -> 192 neurones caches
    nn.Tanh(),                          # le pli (tanh, le meme qu'au chapitre 4)
    nn.Linear(192, vocab_size),         # 192 -> 81 scores
)
```

C'est un MLP. Le tien, en plus grand : 384 entrées au lieu de 2, 192 neurones
cachés au lieu de 16, 81 scores au lieu de 2, tanh au lieu de ReLU. Comptons ses
paramètres et retrouvons, exactement, les 91 497 nombres du chapitre 1.


In [ ]:
embedding = 81 * 24                 # la table de traduction (chapitre 8)
couche_1 = 384 * 192 + 192          # poids + biais de la couche cachee
couche_2 = 192 * 81 + 81            # poids + biais de la couche de sortie

total = embedding + couche_1 + couche_2
print(f"embedding : {embedding:6d}")
print(f"couche 1  : {couche_1:6d}")
print(f"couche 2  : {couche_2:6d}")
print(f"total     : {total:6d}")
assert total == 91497, "on doit retomber exactement sur les 91 497 nombres du chapitre 1"
print("Les 91 497 nombres de MiniLM, retrouves au parametre pres.")


## Exercices

À toi de jouer : trois exercices, du plus simple (●) au plus costaud (●●●),
les trois gestes fondamentaux du chapitre. Chaque cellule marquée `# TODO(toi)`
contient un trou ; complète-le, puis exécute la cellule de validation (`assert`)
qui suit : si elle passe sans erreur, c'est gagné.

**Le pacte « IA débranchée » agit ici** : pas d'IA pour remplir les trous à ta
place. Elle peut t'expliquer une erreur ; les doigts sur le clavier, c'est toi.
Les réponses sont dans le notebook solution, à n'ouvrir qu'après avoir vraiment essayé.

### Exercice 1 · Une couche, de tes mains — niveau ●

Écris le forward d'une couche : le produit matriciel du chapitre 2 (`@`), la
transposée de `W` pour emboîter les dimensions, puis `+ b`. Une seule ligne de
calcul. Rappel des shapes : `entrees (n, e)`, `W (s, e)` (une ligne par
neurone), `b (s,)`, sortie `(n, s)`.

In [ ]:
def forward_couche(entrees, W, b):
    """La couche : chaque ligne de W est la recette d'un neurone.

    entrees : (n, e)   W : (s, e)   b : (s,)   ->   sortie : (n, s)
    """
    # TODO(toi) : une ligne. Le produit matriciel du chapitre 2 (@),
    # la transposee de W (W.T) pour emboiter les dimensions, puis + b.
    ...


# Demonstration sur des valeurs simples, verifiables de tete.
entrees_demo = torch.tensor([[2.0, 3.0],
                             [1.0, -1.0]])
W_demo = torch.tensor([[1.0, 0.0],     # neurone 1 : recopie x1
                       [0.0, 1.0],     # neurone 2 : recopie x2
                       [1.0, 1.0],     # neurone 3 : x1 + x2
                       [0.5, -0.5]])   # neurone 4 : (x1 - x2) / 2
b_demo = torch.tensor([0.0, 0.0, 1.0, 2.0])

sortie_demo = forward_couche(entrees_demo, W_demo, b_demo)
print(f"shape attendue : (2, 4)")
print(sortie_demo)


In [ ]:
# Validation de l'exercice 1 : les quatre recettes, calculees a la main.
attendu = torch.tensor([[2.0, 3.0, 6.0, 1.5],
                        [1.0, -1.0, 1.0, 3.0]])
assert sortie_demo is not None, "forward_couche ne renvoie rien : il manque le return ?"
assert tuple(sortie_demo.shape) == (2, 4), f"shape {tuple(sortie_demo.shape)} au lieu de (2, 4) : W est-elle bien transposee ?"
assert torch.allclose(sortie_demo, attendu), "les valeurs ne collent pas : verifie entrees @ W.T + b"
print("Exercice 1 valide : ta couche calcule juste.")


### Exercice 2 · La boucle d'entraînement en cinq lignes — niveau ●●

Le refrain du livre, de tête cette fois : forward, loss, `zero_grad()`,
`backward()`, `step()`, dans cet ordre. Repars d'un étage linéaire tout neuf et
entraîne-le sur le carnet : tu dois retomber exactement sur le plafond des 63 %
de la leçon.

In [ ]:
torch.manual_seed(1)
modele_exercice = nn.Linear(2, 2)
optimiseur = torch.optim.SGD(modele_exercice.parameters(), lr=0.2)

for etape in range(10001):
    # TODO(toi) : les cinq lignes du refrain, dans l'ordre.
    # 1. logits = ...                    (le forward : modele_exercice applique aux destinations)
    # 2. loss = ...                      (F.cross_entropy entre les logits et les cibles)
    # 3. ...                             (remise a zero des gradients de l'optimiseur)
    # 4. ...                             (le retour : les gradients de la loss)
    # 5. ...                             (le pas de descente de l'optimiseur)
    ...
    if etape % 2000 == 0:
        print(f"etape {etape:5d} | loss = {loss.item():.4f}")

In [ ]:
# Validation de l'exercice 2 : la loss a fondu, le modele fait mieux que le hasard.
loss_exercice = F.cross_entropy(modele_exercice(destinations), cibles).item()
acc_exercice = accuracy(modele_exercice)
print(f"loss finale : {loss_exercice:.4f} | accuracy : {acc_exercice:.3f}")

assert loss_exercice < 0.65, "la loss n'a pas assez fondu : les 5 lignes sont-elles dans le bon ordre ?"
assert 0.55 < acc_exercice < 0.72, "l'accuracy attendue est ~0.63 : relis la boucle"
print("Exercice 2 valide : ta boucle entraine. Meme plafond de 63 % que la lecon, comme prevu.")

### Exercice 3 · ReLU et le forward d'un MLP — niveau ●●●

Écris `relu_maison` (`max(0, x)`, appliqué élément par élément : le pli), puis
le forward complet d'un MLP : couche, pli, couche. Réutilise ta `forward_couche`
de l'exercice 1.

In [ ]:
def relu_maison(t):
    """max(0, x), applique element par element : le pli."""
    # TODO(toi) : une ligne. torch.clamp(t, min=0.0) ou t * (t > 0), au choix.
    ...


def forward_mlp(entrees, W1, b1, W2, b2):
    """Couche -> pli -> couche : le plus petit MLP du monde."""
    # TODO(toi) : deux lignes.
    # 1. cachee = le pli applique a forward_couche(entrees, W1, b1)
    # 2. return forward_couche(cachee, W2, b2)
    ...


# Demonstration, verifiable a la main.
e = torch.tensor([[1.0, 2.0], [-2.0, 0.5]])
W1_demo = torch.tensor([[1.0, 0.0], [-1.0, 1.0]])
b1_demo = torch.tensor([0.0, 1.0])
W2_demo = torch.tensor([[2.0, -1.0]])
b2_demo = torch.tensor([0.5])
sortie_mlp = forward_mlp(e, W1_demo, b1_demo, W2_demo, b2_demo)
print(sortie_mlp)


In [ ]:
# Validation de l'exercice 3, pas a pas puis de bout en bout.
assert torch.equal(relu_maison(torch.tensor([-3.0, 0.0, 2.5])), torch.tensor([0.0, 0.0, 2.5])), \
    "relu_maison doit ecraser les negatifs a zero et laisser passer le reste"

# A la main : pre-activation [[1, 2], [-2, 3.5]] -> relu [[1, 2], [0, 3.5]]
# -> sortie [2*1 - 1*2 + 0.5, 2*0 - 1*3.5 + 0.5] = [0.5, -3.0]
attendu_mlp = torch.tensor([[0.5], [-3.0]])
assert sortie_mlp is not None and torch.allclose(sortie_mlp, attendu_mlp), \
    "le forward du MLP ne colle pas : couche -> pli -> couche, dans cet ordre"
print("Exercice 3 valide : tu sais ecrire un MLP a la main.")


---

## Verdict

Trois validations vertes : tu sais construire une couche, la plier, l'empiler,
et entraîner le tout avec la boucle en cinq lignes. Ton réseau vient de retrouver
deux zones de Cotonou qu'on ne lui avait jamais dessinées : il a appris une
frontière courbe à partir d'exemples, exactement comme MiniLM apprenait à
prédire un caractère.

Mais ton réseau mange des **nombres**. Deux coordonnées en kilomètres, ça se
donne tel quel ; un texte, non. Au **chapitre 7**, on découpe le langage en
morceaux (la tokenisation, ton propre BPE) ; au **chapitre 8**, on apprend à
donner du sens à ces morceaux (les embeddings), et le MLP de ce chapitre
deviendra un vrai modèle de langage.